    # GNN MAPP: All Modules in One Notebook

    This notebook concatenates all project modules into one Colab-friendly workflow.
    Run cells top-to-bottom to define every module and do a quick end-to-end training smoke test.
    

    ## 1) Colab Setup

    Install runtime dependencies once per session.
    

In [87]:
%pip -q install vmas matplotlib torch


## 2) Shared Imports

In [88]:
import os
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.optim import Adam
from torch.distributions import Categorical
import vmas


## 3) `utils.py`

In [89]:
def build_adj(agent_pos, r_comm):
    """
    Build a degree-normalised adjacency matrix from agent positions.
    agent_pos : (N, 2) tensor of absolute x/y positions
    r_comm    : communication radius (scalar)
    Returns   : (N, N) normalised adjacency tensor
    """
    diff = agent_pos.unsqueeze(1) - agent_pos.unsqueeze(0)   # (N, N, 2)
    dist = torch.norm(diff, dim=-1)                          # (N, N)
    adj  = (dist <= r_comm).float()
    deg  = adj.sum(dim=1, keepdims=True).clamp(min=1)
    return adj / deg


def _vmas_to_scalar(t):
    """
    VMAS returns tensors with inconsistent shapes depending on the value type:
      - observations : (num_envs, obs_dim)  -> need [0] to get (obs_dim,)
      - rewards      : (num_envs,) OR 0-dim scalar
      - dones        : (num_envs,) OR 0-dim scalar

    This helper extracts a plain Python scalar from any of those shapes safely.
    """
    if t is None:
        return None
    t = t.detach()
    return t.reshape(-1)[0].item() if t.numel() > 0 else 0.0


def _vmas_to_numpy_obs(t):
    """
    Extract a (obs_dim,) numpy array from a VMAS obs tensor.
    Handles both (num_envs, obs_dim) and (obs_dim,) shapes.
    """
    t = t.detach()
    if t.dim() == 2:
        return t[0].cpu().numpy()    # (num_envs, obs_dim) -> (obs_dim,)
    return t.cpu().numpy()           # (obs_dim,) already


class VMASAdapter:
    """
    Thin wrapper that makes a VMAS environment behave like the PettingZoo
    parallel API expected by GNNTrainer:
        env.possible_agents          -> list[str]
        env.reset(seed)              -> (obs_dict, info)
        env.step(actions_dict)       -> (obs_dict, rew_dict, done_dict, trunc_dict, info)
        env.get_agent_positions(dev) -> (N, 2) position tensor  [VMAS-specific]
        env.obs_dim                  -> int   [inferred on first reset]

    Design choices:
      - continuous_actions=False : 5-action discrete space matching MPE
        (no-op / left / right / up / down)
      - Agent positions for graph construction are read directly from the VMAS
        world state (get_agent_positions), not from obs indices, so graph
        topology is unambiguous regardless of obs layout.
    """

    ACTION_DIM = 5  # no-op + 4 cardinal directions

    def __init__(self, n_agents, max_steps, device, seed=None):
        self.n_agents        = n_agents
        self._device         = device
        self._max_steps      = max_steps
        self._step_count     = 0
        self.obs_dim         = None   # set after first reset()

        self.possible_agents = [f"agent_{i}" for i in range(n_agents)]
        self.agents          = list(self.possible_agents)

        self._env = vmas.make_env(
            scenario="navigation",
            num_envs=1,
            n_agents=n_agents,
            device=device,
            continuous_actions=False,
            seed=seed,
        )

    # ------------------------------------------------------------------
    def reset(self, seed=None):
        if seed is not None:
            obs_list = self._env.reset(seed=seed)
        else:
            obs_list = self._env.reset()
        self._step_count = 0

        obs_dict = {
            aid: _vmas_to_numpy_obs(obs_list[i])
            for i, aid in enumerate(self.possible_agents)
        }
        # Infer obs_dim from actual env output — never hardcode
        self.obs_dim = obs_dict[self.possible_agents[0]].shape[0]
        return obs_dict, {}

    # ------------------------------------------------------------------
    def step(self, actions_dict):
        action_list = [
            torch.tensor([actions_dict[aid]], device=self._device)
            for aid in self.possible_agents
        ]

        # VMAS step returns (obs_list, rew_list, done_list, info)
        obs_list, rew_list, done, _info = self._env.step(action_list)
        self._step_count += 1
        truncated = self._step_count >= self._max_steps
        obs_dict, rew_dict, done_dict, trunc_dict = {}, {}, {}, {}
        for i, aid in enumerate(self.possible_agents):
            obs_dict[aid]   = _vmas_to_numpy_obs(obs_list[i])
            rew_dict[aid]   = float(_vmas_to_scalar(rew_list[i]))
            # _vmas_to_scalar handles both 0-dim scalar and (1,) shaped done tensors
            agent_done      = bool(done[0])
            done_dict[aid]  = agent_done or truncated
            trunc_dict[aid] = truncated

        return obs_dict, rew_dict, done_dict, trunc_dict, {}

    # ------------------------------------------------------------------
    def get_agent_positions(self, device):
        """
        Read absolute agent positions directly from the VMAS world state.
        Returns: (N, 2) float32 tensor on `device`
        """
        pos = torch.stack(
            [self._env.agents[i].state.pos.reshape(-1)[:2]
             for i in range(self.n_agents)]
        )  # (N, 2) — reshape(-1) guards against (1,2) or (2,) shapes
        return pos.to(device=device, dtype=torch.float32)

    def close(self):
        pass








## 4) `observation.py`

In [90]:
class ObservationEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(ObservationEncoder, self).__init__()

        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, output_dim)

        self._init_weights()

    def _init_weights(self):
        for layer in [self.fc1, self.fc2, self.fc3]:
            nn.init.orthogonal_(layer.weight, gain=2**0.5)
            nn.init.constant_(layer.bias, 0.0)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)


## 5) `graphConv.py`

In [91]:
class GraphConv(nn.Module):
    def __init__(self, F, G, K):
        super(GraphConv, self).__init__()
        self.F = F
        self.G = G
        self.K = K

        self.weights = nn.ParameterList(
            [nn.Parameter(torch.empty(F, G)) for _ in range(K)]
        )
        for p in self.weights:
            nn.init.xavier_uniform_(p)

    def forward(self, X, S):
        """
        X : (N, F)  node feature matrix
        S : (N, N)  normalised adjacency (shift operator)
        Returns (N, G) aggregated features.
        Implements: Z = sum_{k=0}^{K-1} S^k X W_k
        """
        Z     = X
        accum = X.new_zeros(*X.shape[:-1], self.G)
        for k in range(self.K):
            accum += torch.matmul(Z, self.weights[k])
            Z = torch.matmul(S, Z)
        return accum


## 6) `action.py`

In [92]:
class ActionHead(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(ActionHead, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)
        self._init_weights()

    def _init_weights(self):
        nn.init.orthogonal_(self.fc1.weight, gain=2**0.5)
        nn.init.constant_(self.fc1.bias, 0.0)
        nn.init.orthogonal_(self.fc2.weight, gain=0.01)
        nn.init.constant_(self.fc2.bias, 0.0)

    def forward(self, G):
        return self.fc2(F.relu(self.fc1(G)))


## 7) `commPolicy.py`

In [93]:
class CommPolicy(nn.Module):
    def __init__(self, obs_dim, hidden_dim, action_dim, F, G, K):
        super(CommPolicy, self).__init__()
        self.obsEncoder = ObservationEncoder(obs_dim, hidden_dim, F)
        self.graphConv  = GraphConv(F, G, K)
        self.actionHead = ActionHead(G, hidden_dim, action_dim)

    def _to_tensor(self, x, dtype=torch.float32):
        device = next(self.parameters()).device
        if not torch.is_tensor(x):
            return torch.as_tensor(x, dtype=dtype, device=device)
        return x.to(device=device, dtype=dtype)

    def forward(self, obs, S):
        obs = self._to_tensor(obs)
        S   = self._to_tensor(S)
        return self.actionHead(self.graphConv(self.obsEncoder(obs), S))

    def get_actions(self, obs, S):
        dist   = Categorical(logits=self.forward(obs, S))
        action = dist.sample()
        return action, dist.log_prob(action), dist.entropy()

    def evaluate_actions(self, obs, S, actions):
        logits  = self.forward(obs, S)
        device  = logits.device
        actions = actions.to(device=device, dtype=torch.long)
        dist    = Categorical(logits=logits)
        return dist.log_prob(actions), dist.entropy(), logits


## 8) `gppoAgent.py`

In [94]:
class CriticNetwork(nn.Module):
    def __init__(self, obs_dim, hidden_dim, device=None):
        super(CriticNetwork, self).__init__()
        self.fc1    = nn.Linear(obs_dim, hidden_dim)
        self.fc2    = nn.Linear(hidden_dim, hidden_dim)
        self.fc3    = nn.Linear(hidden_dim, 1)
        self.device = device
        self._init_weights()
        self.to(self.device)

    def _init_weights(self):
        nn.init.orthogonal_(self.fc1.weight, gain=2**0.5)
        nn.init.orthogonal_(self.fc2.weight, gain=2**0.5)
        nn.init.orthogonal_(self.fc3.weight, gain=1.0)
        for layer in [self.fc1, self.fc2, self.fc3]:
            nn.init.constant_(layer.bias, 0.0)

    def forward(self, obs):
        if not torch.is_tensor(obs):
            obs = torch.tensor(obs, dtype=torch.float32, device=self.device)
        else:
            obs = obs.to(self.device, dtype=torch.float32)
        return self.fc3(F.relu(self.fc2(F.relu(self.fc1(obs)))))


## 9) `rolloutBuffer.py`

In [95]:
class GNNRolloutBuffer:
    def __init__(self, gamma, gae_lambda, device):
        self.gamma      = gamma
        self.gae_lambda = gae_lambda
        self.device     = device
        self.clear()

    def add_timestep(self, obs, actions, rewards, dones, log_probs, values, A):
        self.obs.append(obs)
        self.actions.append(actions)
        self.rewards.append(rewards)
        self.dones.append(dones)
        self.log_probs.append(log_probs)
        self.values.append(values)
        self.adj.append(A)

    def compute_advantages(self, last_values):
        N           = len(self.obs[0])
        buffer_size = len(self.obs)
        rewards  = torch.stack(self.rewards).to(device=self.device, dtype=torch.float32)
        values   = torch.stack(self.values).to(device=self.device, dtype=torch.float32)
        dones    = torch.stack(self.dones).to(device=self.device, dtype=torch.float32)

        advantages = torch.zeros((buffer_size, N), dtype=torch.float32, device=self.device)
        last_gae   = torch.zeros(N, dtype=torch.float32, device=self.device)

        if not torch.is_tensor(last_values):
            last_values = torch.as_tensor(last_values, dtype=torch.float32, device=self.device)
        else:
            last_values = last_values.to(device=self.device, dtype=torch.float32)

        for t in reversed(range(buffer_size)):
            next_val  = last_values if t == buffer_size - 1 else values[t + 1]
            delta     = rewards[t] + self.gamma * (1 - dones[t]) * next_val - values[t]
            last_gae  = delta + self.gamma * self.gae_lambda * (1 - dones[t]) * last_gae
            advantages[t] = last_gae

        self.advantages = advantages
        self.returns    = advantages + values

    def get_batches(self, B):
        perm     = torch.randperm(len(self.obs), device=self.device)
        obs      = torch.stack(self.obs).to(device=self.device, dtype=torch.float32)
        actions  = torch.stack(self.actions).to(device=self.device, dtype=torch.long)
        lp       = torch.stack(self.log_probs).to(device=self.device, dtype=torch.float32)
        adj      = torch.stack(self.adj).to(device=self.device, dtype=torch.float32)
        adv      = (self.advantages - self.advantages.mean()) / (self.advantages.std() + 1e-8)

        for idx in perm.split(B):
            yield obs[idx], actions[idx], lp[idx], adv[idx], self.returns[idx], adj[idx]

    def clear(self):
        self.obs, self.actions, self.rewards  = [], [], []
        self.dones, self.log_probs, self.values, self.adj = [], [], [], []
        self.advantages = self.returns = None


## 10) `trainer.py`

In [96]:
class GNNTrainer:
    def __init__(
        self,
        num_agents,
        env,
        obs_dim,
        hidden_dim,
        action_dim,
        F, G, K,
        lr, gamma, gae_lambda,
        clip_eps, value_coef, entropy_coef,
        device,
    ):
        self.device      = device
        self.num_agents  = num_agents
        self.agent_ids   = sorted(env.possible_agents)
        self.clip_eps    = clip_eps
        self.value_coef  = value_coef
        self.entropy_coef = entropy_coef

        self.comm_policy = CommPolicy(
            obs_dim=obs_dim, hidden_dim=hidden_dim,
            action_dim=action_dim, F=F, G=G, K=K
        ).to(self.device)
        self.comm_optim = Adam(self.comm_policy.parameters(), lr=lr)

        self.critic       = CriticNetwork(obs_dim=obs_dim, hidden_dim=hidden_dim, device=self.device)
        self.critic_optim = Adam(self.critic.parameters(), lr=lr)

        self.buffer  = GNNRolloutBuffer(gamma=gamma, gae_lambda=gae_lambda, device=self.device)
        self.env     = env
        self._running_episode_return = 0.0
        self.metrics_history = {
            "policy_loss": [], "value_loss": [], "entropy": [],
            "mean_bellman_error": [], "mean_episode_return": [], "mean_episode_rewards": [],
        }

        # FIXED (Bug 6): trainer no longer calls env.reset() internally.
        # The caller resets the env (with seed) and passes the initial obs in.
        # Use set_initial_obs() after constructing the trainer.

    def set_initial_obs(self, obs_dict):
        """Call this once after env.reset(seed=...) to prime the trainer."""
        self.current_obs = torch.stack(
            [torch.from_numpy(obs_dict[a]) for a in self.agent_ids]
        ).to(device=self.device, dtype=torch.float32)

    def _safe_mean(self, values):
        return float(sum(values) / len(values)) if values else 0.0

    def collect_rollouts(self, num_steps, r_comm=1.0):
        obs_tensor = self.current_obs
        step_mean_rewards, completed_episode_returns = [], []

        for _ in range(num_steps):
            # FIXED (Bug 4): get ground-truth positions from the adapter,
            # not from a fragile obs-index slice
            agent_pos = self.env.get_agent_positions(self.device)
            S = build_adj(agent_pos, r_comm)

            actions, log_probs, entropy = self.comm_policy.get_actions(obs=obs_tensor, S=S)
            values = self.critic(obs_tensor).detach().squeeze(-1)  # (N,)

            actions_dict = {a_id: actions[i].cpu().item() for i, a_id in enumerate(self.agent_ids)}
            next_obs, rewards, dones, truncs, _ = self.env.step(actions_dict)

            rewards_tensor = torch.tensor(
                [rewards[a] for a in self.agent_ids], dtype=torch.float32, device=self.device
            )
            dones_tensor = torch.tensor(
                [dones[a] for a in self.agent_ids], dtype=torch.float32, device=self.device
            )

            self.buffer.add_timestep(
                obs=obs_tensor.detach(), actions=actions.detach(),
                rewards=rewards_tensor, dones=dones_tensor,
                log_probs=log_probs.detach(), values=values, A=S.detach(),
            )

            step_mean_rewards.append(rewards_tensor.mean().item())
            self._running_episode_return += rewards_tensor.mean().item()

            if all(dones.values()) or all(truncs.values()):
                completed_episode_returns.append(self._running_episode_return)
                self._running_episode_return = 0.0
                obs_reset, _ = self.env.reset()
                obs_tensor = torch.stack(
                    [torch.from_numpy(obs_reset[a]) for a in self.agent_ids]
                ).to(device=self.device, dtype=torch.float32)
            else:
                obs_tensor = torch.stack(
                    [torch.from_numpy(next_obs[a]) for a in self.agent_ids]
                ).to(device=self.device, dtype=torch.float32)

            self.current_obs = obs_tensor

        rollout_metrics = {
            "mean_episode_return":  self._safe_mean(completed_episode_returns)
                                    if completed_episode_returns
                                    else float(self._running_episode_return),
            "mean_episode_rewards": self._safe_mean(step_mean_rewards),
        }
        self.metrics_history["mean_episode_return"].append(rollout_metrics["mean_episode_return"])
        self.metrics_history["mean_episode_rewards"].append(rollout_metrics["mean_episode_rewards"])
        return obs_tensor, rollout_metrics

    def update(self, last_obs, num_epochs=10, B=64):
        with torch.no_grad():
            last_values = self.critic(
                last_obs.to(device=self.device, dtype=torch.float32)
            ).squeeze(-1)
        self.buffer.compute_advantages(last_values=last_values)

        policy_losses, entropies, value_losses, bellman_errors = [], [], [], []

        for _ in range(num_epochs):
            for obs, actions, old_lp, advantages, returns, A in self.buffer.get_batches(B):
                # ---- policy update ----
                new_lp, entropy, _ = self.comm_policy.evaluate_actions(obs, A, actions)
                ratio  = torch.exp(new_lp - old_lp)
                surr1  = ratio * advantages
                surr2  = torch.clamp(ratio, 1 - self.clip_eps, 1 + self.clip_eps) * advantages
                p_loss = -torch.min(surr1, surr2).mean() - self.entropy_coef * entropy.mean()

                self.comm_optim.zero_grad()
                p_loss.backward()
                nn.utils.clip_grad_norm_(self.comm_policy.parameters(), max_norm=0.5)
                self.comm_optim.step()
                policy_losses.append(p_loss.item())
                entropies.append(entropy.mean().item())

                # ---- value update ----
                B_size, N, obs_d = obs.shape
                flat_obs     = obs.reshape(B_size * N, obs_d)
                flat_returns = returns.reshape(B_size * N)
                pred_vals    = self.critic(flat_obs).squeeze(-1)
                v_loss       = F.mse_loss(pred_vals, flat_returns)
                bellman_err  = (flat_returns - pred_vals).abs().mean()

                self.critic_optim.zero_grad()
                v_loss.backward()
                nn.utils.clip_grad_norm_(self.critic.parameters(), max_norm=0.5)
                self.critic_optim.step()
                value_losses.append(v_loss.item())
                bellman_errors.append(bellman_err.item())

        self.buffer.clear()
        update_metrics = {
            "policy_loss":        self._safe_mean(policy_losses),
            "value_loss":         self._safe_mean(value_losses),
            "entropy":            self._safe_mean(entropies),
            "mean_bellman_error": self._safe_mean(bellman_errors),
        }
        for k, v in update_metrics.items():
            self.metrics_history[k].append(v)
        return update_metrics


## 11) Quick Module Smoke Tests

In [97]:
torch.manual_seed(0)

# --- standalone architecture tests (no env needed) ---
N_agents  = 3
obs_dim   = 18   # arbitrary — just tests tensor flow through each module
hidden_dim  = 64
action_dim  = VMASAdapter.ACTION_DIM
F_dim = G_dim = 64
K_hops = 2

sample_obs = torch.randn(N_agents, obs_dim)
sample_pos = torch.randn(N_agents, 2)
sample_adj = build_adj(sample_pos, r_comm=1.5)

enc = ObservationEncoder(obs_dim, hidden_dim, F_dim)
print("ObservationEncoder:", enc(sample_obs).shape)             # (3, 64)

gc  = GraphConv(F_dim, G_dim, K_hops)
print("GraphConv:", gc(enc(sample_obs), sample_adj).shape)      # (3, 64)

ah  = ActionHead(G_dim, hidden_dim, action_dim)
print("ActionHead:", ah(gc(enc(sample_obs), sample_adj)).shape) # (3, 5)

pol = CommPolicy(obs_dim, hidden_dim, action_dim, F_dim, G_dim, K_hops)
acts, logp, ent = pol.get_actions(sample_obs, sample_adj)
print("CommPolicy actions / logp / entropy:", acts.shape, logp.shape, ent.shape)

crit = CriticNetwork(obs_dim, hidden_dim, device="cpu")
print("CriticNetwork:", crit(sample_obs).shape)                 # (3, 1)

# --- VMAS adapter sanity check ---
print("\n--- VMASAdapter sanity check ---")
_test_env = VMASAdapter(n_agents=3, max_steps=25, device="cpu", seed=0)
_obs, _   = _test_env.reset(seed=0)
print(f"obs_dim inferred from env: {_test_env.obs_dim}")        # e.g. 12 for 3 agents
print(f"sample obs shape: {_obs["agent_0"].shape}")
_pos = _test_env.get_agent_positions("cpu")
print(f"agent positions shape: {_pos.shape}")                   # (3, 2)
print(f"agent positions:\n{_pos}")
_adj = build_adj(_pos, r_comm=1.0)
print(f"adjacency matrix:\n{_adj.round(decimals=2)}")
_test_env.close()
print("\nAll smoke tests passed.")


ObservationEncoder: torch.Size([3, 64])
GraphConv: torch.Size([3, 64])
ActionHead: torch.Size([3, 5])
CommPolicy actions / logp / entropy: torch.Size([3]) torch.Size([3]) torch.Size([3])
CriticNetwork: torch.Size([3, 1])

--- VMASAdapter sanity check ---
obs_dim inferred from env: 18
sample obs shape: (18,)
agent positions shape: torch.Size([3, 2])
agent positions:
tensor([[-0.0075,  0.5364],
        [-0.8230, -0.7359],
        [-0.3852,  0.2682]])
adjacency matrix:
tensor([[0.5000, 0.0000, 0.5000],
        [0.0000, 1.0000, 0.0000],
        [0.5000, 0.0000, 0.5000]])

All smoke tests passed.


## 12) Experiment Sweep: `K` and `r_comm`

Runs a full grid over `K in [1, 3, 5]` and `r_comm in [0.5, 1.0, 1.5, 2.0]`
with `TOTAL_TIMESTEPS = 1_000_000`, `ROLLOUT_LENGTH = 2048`, `NUM_AGENTS = 10`.

`obs_dim` is inferred automatically from the first `env.reset()` so no
hardcoded dimension mismatch is possible.


In [ ]:
# ============================================================
# Hyperparameters
# ============================================================
NUM_AGENTS    = 10
MAX_CYCLES    = 100
action_dim    = VMASAdapter.ACTION_DIM   # 5

F_dim = G_dim = hidden_dim = 64

lr            = 3e-4
gamma         = 0.99
gae_lambda    = 0.85
clip_eps      = 0.2
value_coef    = 0.5
entropy_coef  = 0.01

TOTAL_TIMESTEPS = 1_000_000
ROLLOUT_LENGTH  = 2048
BATCH_SIZE      = 64
NUM_EPOCHS      = 10

K_VALUES       = [1, 3, 5]
R_COMM_VALUES  = [0.5, 1.0, 1.5, 2.0]
SEED           = 42
LOG_EVERY      = 10

OUTPUT_DIR = "outputs/experiments_k_r"
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# ============================================================
# FIXED (Bug 3): infer obs_dim from the environment, never hardcode it.
# VMAS navigation obs = vel(2) + abs_pos(2) + goal_rel(2) + other_agents_rel(2*(N-1))
# For N=10: obs_dim = 2+2+2+18 = 24. This is detected automatically below.
# ============================================================
_probe_env = VMASAdapter(n_agents=NUM_AGENTS, max_steps=MAX_CYCLES, device=device, seed=SEED)
_probe_obs, _ = _probe_env.reset(seed=SEED)
obs_dim = _probe_env.obs_dim
_probe_env.close()
print(f"obs_dim (inferred from env): {obs_dim}")
print(f"Sweep: K={K_VALUES}, r_comm={R_COMM_VALUES}")

# ============================================================
# FIXED (Bug 2): helpers and state moved HERE, outside the commented block
# ============================================================
metrics_to_plot = [
    "policy_loss", "value_loss", "entropy",
    "mean_bellman_error", "mean_episode_return", "mean_episode_rewards",
]

def plot_metrics(metrics_history, title, save_path):
    fig, axes = plt.subplots(3, 2, figsize=(14, 12))
    for ax, name in zip(axes.flatten(), metrics_to_plot):
        vals = metrics_history.get(name, [])
        ax.plot(range(1, len(vals) + 1), vals, linewidth=1.8)
        ax.set_title(name); ax.set_xlabel("Iteration")
        ax.set_ylabel(name); ax.grid(True, alpha=0.3)
    fig.suptitle(title)
    plt.tight_layout()
    fig.savefig(save_path, dpi=180, bbox_inches="tight")
    plt.close(fig)

results      = {}
summary_rows = []

# ============================================================
# Main sweep
# ============================================================
for K_hops in K_VALUES:
    for R_COMM in R_COMM_VALUES:
        # FIXED (Bug 1): config_name was missing from the VMAS loop
        config_name = f"K={K_hops}, r_comm={R_COMM:.2f}"
        print(f"\n=== Running {config_name} ===")

        torch.manual_seed(SEED)
        np.random.seed(SEED)

        env = VMASAdapter(
            n_agents=NUM_AGENTS, max_steps=MAX_CYCLES,
            device=device, seed=SEED,
        )


        # FIXED (Bug 6): reset here with seed, then hand obs to trainer.
        # Trainer no longer calls env.reset() internally.
        obs, _ = env.reset(seed=SEED)

        trainer = GNNTrainer(
            num_agents=NUM_AGENTS, env=env, obs_dim=obs_dim,
            hidden_dim=hidden_dim, action_dim=action_dim,
            F=F_dim, G=G_dim, K=K_hops,
            lr=lr, gamma=gamma, gae_lambda=gae_lambda,
            clip_eps=clip_eps, value_coef=value_coef, entropy_coef=entropy_coef,
            device=device,
        )
        trainer.set_initial_obs(obs)   # prime with the seeded initial observation

        steps_done = 0
        iteration  = 0

        while steps_done < TOTAL_TIMESTEPS:
            rollout_steps = min(ROLLOUT_LENGTH, TOTAL_TIMESTEPS - steps_done)
            last_obs, rollout_metrics = trainer.collect_rollouts(
                num_steps=rollout_steps, r_comm=R_COMM
            )
            update_metrics = trainer.update(last_obs, num_epochs=NUM_EPOCHS, B=BATCH_SIZE)

            steps_done += rollout_steps
            iteration  += 1

            should_log = (iteration == 1 or iteration % LOG_EVERY == 0
                          or steps_done == TOTAL_TIMESTEPS)
            if should_log:
                print(
                    f"[{config_name}] Iter {iteration:4d} | "
                    f"steps={steps_done:>8}/{TOTAL_TIMESTEPS} | "
                    f"pi_loss={update_metrics['policy_loss']:.4f} | "
                    f"v_loss={update_metrics['value_loss']:.4f} | "
                    f"ent={update_metrics['entropy']:.4f} | "
                    f"bellman={update_metrics['mean_bellman_error']:.4f} | "
                    f"ep_ret={rollout_metrics['mean_episode_return']:.4f} | "
                    f"ep_rew={rollout_metrics['mean_episode_rewards']:.4f}"
                )

        results[(K_hops, R_COMM)] = trainer.metrics_history
        final_return = trainer.metrics_history["mean_episode_return"][-1]
        final_reward = trainer.metrics_history["mean_episode_rewards"][-1]
        summary_rows.append((K_hops, R_COMM, final_return, final_reward))

        plot_path = os.path.join(OUTPUT_DIR, f"metrics_K{K_hops}_r{R_COMM:.2f}.png")
        plot_metrics(trainer.metrics_history, title=f"Training Metrics ({config_name})", save_path=plot_path)
        print(f"Saved: {plot_path}")
        env.close()


# ============================================================
# Summary heatmap
# ============================================================
return_matrix = np.full((len(K_VALUES), len(R_COMM_VALUES)), np.nan, dtype=np.float32)
for K_hops, R_COMM, final_return, _ in summary_rows:
    i = K_VALUES.index(K_hops)
    j = R_COMM_VALUES.index(R_COMM)
    return_matrix[i, j] = final_return

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(return_matrix, cmap="viridis")
ax.set_xticks(range(len(R_COMM_VALUES))); ax.set_xticklabels([str(v) for v in R_COMM_VALUES])
ax.set_yticks(range(len(K_VALUES)));     ax.set_yticklabels([str(v) for v in K_VALUES])
ax.set_xlabel("r_comm"); ax.set_ylabel("K"); ax.set_title("Final Mean Episode Return (VMAS Navigation)")
for i in range(len(K_VALUES)):
    for j in range(len(R_COMM_VALUES)):
        ax.text(j, i, f"{return_matrix[i, j]:.2f}", ha="center", va="center", color="white", fontsize=9)
fig.colorbar(im, ax=ax)
plt.tight_layout()
summary_path = os.path.join(OUTPUT_DIR, "summary_heatmap.png")
plt.savefig(summary_path, dpi=180); plt.show()
print(f"Saved heatmap: {summary_path}")

print("\nFinal summary (K, r_comm, final_return, final_reward):")
for row in summary_rows:
    print(row)


Device: cuda
obs_dim (inferred from env): 18
Sweep: K=[1, 3, 5], r_comm=[0.5, 1.0, 1.5, 2.0]

=== Running K=1, r_comm=0.50 ===
[K=1, r_comm=0.50] Iter    1 | steps=    2048/1000000 | pi_loss=-0.0198 | v_loss=0.2063 | ent=1.6037 | bellman=0.2996 | ep_ret=-9.6313 | ep_rew=-0.0957
[K=1, r_comm=0.50] Iter   10 | steps=   20480/1000000 | pi_loss=-0.0212 | v_loss=0.0304 | ent=1.3069 | bellman=0.1296 | ep_ret=1.6326 | ep_rew=0.0153
